# All the parameters that need to be changed

In [1]:
# Mississippi
state_ab = "ms"

## Data
1. Download all the data in directory "il_data"
2. Eextract them all

In [3]:
data_folder = state_ab + "_data/"
population1_data = "./{}{}_pl2020_b/{}_pl2020_p1_b.shp".format(data_folder, state_ab, state_ab)
population2_data = "./{}{}_pl2020_b/{}_pl2020_p2_b.shp".format(data_folder, state_ab, state_ab)
vap_data =  "./{}{}_pl2020_b/{}_pl2020_p4_b.shp".format(data_folder, state_ab, state_ab)
vest20_data = "./{}{}_vest_20/{}_vest_20.shp".format(data_folder, state_ab, state_ab)
vest19_data = "./{}{}_vest_19/{}_vest_19.shp".format(data_folder, state_ab, state_ab)
vest18_data = "./{}{}_vest_18/{}_vest_18.shp".format(data_folder, state_ab, state_ab)
vest16_data = "./{}{}_vest_16/{}_vest_16.shp".format(data_folder, state_ab, state_ab)
cd_data = "./{}{}_cong_adopted_2022/MS_USCongDists_2022.shp".format(data_folder, state_ab)
send_data = "./{}{}_sldu_adopted_2022/MS_SenateDists_Mar2022.shp".format(data_folder, state_ab)
hdist_data = "./{}{}_sldl_adopted_2022/MS_HouseDists_Mar292022.shp".format(data_folder, state_ab)

## Parameters that needs to be manually checked

### base vest data
start_col = 5\
vest_base_data = vest20\
year = '20'

### district data
district column name of cong_df, send, hdist when calling add_dist()

# Program starts

In [4]:
import pandas as pd
import geopandas as gpd
import maup
import time
from maup import smart_repair
from gerrychain import Graph
import os

maup.progress.enabled = True

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [5]:
import warnings
warnings.filterwarnings("ignore")

In [29]:
def do_smart_repair(df):
    # change it to the UTM it needs for smart_repair
    df = df.to_crs(df.estimate_utm_crs())
    df = smart_repair(df, snap_precision=8)
    
    # check maup doctor again to see if smart repair works
    if maup.doctor(df) == True:
        # change it back to this UTM for this data
        df = df.to_crs('EPSG:4269')
    else:
        raise Exception('maup.doctor failed')
    
    return df

In [7]:
def add_district(dist_df, dist_name, election_df, col_name):
    # check if it needs to be smart_repair
    if maup.doctor(dist_df) != True:
        dist_df = do_smart_repair(dist_df)

    election_df = gpd.GeoDataFrame(election_df, crs="EPSG:4269")

    # assigne the pricincts
    precincts_to_district_assignment = maup.assign(election_df.geometry, dist_df.geometry)
    election_df[dist_name] = precincts_to_district_assignment
    for precinct_index in range(len(election_df)):
        election_df.at[precinct_index, dist_name] = dist_df.at[election_df.at[precinct_index, dist_name], col_name]

    return election_df

In [8]:
def rename(original, year):
    party = original[6]
    if party == 'R' or party == 'D':
        return original[3:6] + year + original[6]
    else:
        return original[3:6] + year + 'O'

In [9]:
def check_population(population, df):
    pop_check = pd.DataFrame({
        'pop_col': pop_col,
        'population_df': population[pop_col].sum(), 
        'vest_base': df[pop_col].sum(),
        'equal': [x == y for x, y in zip(population[pop_col].sum(), df[pop_col].sum())]
    })
    if pop_check['equal'].mean() < 1:
        print(pop_check)
        raise Exception("population doesn't agree")

    else:
        print("population agrees")

In [10]:
def add_vest(vest, df, year, population, start_col):    
     # check if it needs to be smart_repair
    if maup.doctor(vest) != True:
        vest = do_smart_repair(vest)
    
    # rename the columns
    original_col = vest.columns[start_col:-1]
    new_col = [rename(i, year) for i in original_col]
    rename_dict = dict(zip(original_col, new_col))
    vest = vest.rename(columns=rename_dict)
    vest = vest.groupby(level=0, axis=1).sum() # combine all the other party's vote into columns with sufix "O"
    col_name = list(set(new_col))
    col_name.sort()
    
    # make the blocks from precincts by weight
    vest = gpd.GeoDataFrame(vest, crs="EPSG:4269")
    election_in_block = population[["VAP", 'geometry']] # population_df is in block scale
    blocks_to_precincts_assignment = maup.assign(election_in_block.geometry, vest.geometry)
    weights = election_in_block["VAP"] / blocks_to_precincts_assignment.map(election_in_block["VAP"].groupby(blocks_to_precincts_assignment).sum())
    weights = weights.fillna(0)
    prorated = maup.prorate(blocks_to_precincts_assignment, vest[col_name], weights)
    election_in_block[col_name] = prorated
    
    # assign blocks to precincts
    election_in_block = gpd.GeoDataFrame(election_in_block, crs="EPSG:4269")
    df = gpd.GeoDataFrame(df, crs="EPSG:4269")
    block_to_pricinct_assginment = maup.assign(election_in_block.geometry, df.geometry)
    df[col_name] = election_in_block[col_name].groupby(block_to_pricinct_assginment).sum()
    df = df.groupby(level=0, axis=1).sum()
    
    # check if population agrees
    check_population(population, df)
    
    return df

## Read the census data

In [11]:
population1_df = gpd.read_file(population1_data)
population2_df = gpd.read_file(population2_data)
vap_df = gpd.read_file(vap_data)

In [12]:
population2_df = population2_df.drop(columns=['SUMLEV', 'LOGRECNO', 'GEOID', 'COUNTY', 'geometry'])
vap_df = vap_df.drop(columns=['SUMLEV', 'LOGRECNO', 'GEOID', 'COUNTY', 'geometry'])

In [13]:
population_df = pd.merge(population1_df, population2_df, on='GEOID20')
population_df = pd.merge(population_df, vap_df, on='GEOID20')

In [14]:
rename_dict = {'P0020001': 'TOTPOP', 'P0020002': 'HISP', 'P0020005': 'NH_WHITE', 'P0020006': 'NH_BLACK', 'P0020007': 'NH_AMIN',
               'P0020008': 'NH_ASIAN', 'P0020009': 'NH_NHPI', 'P0020010': 'NH_OTHER', 'P0020011': 'NH_2MORE',
               'P0040001': 'VAP', 'P0040002': 'HVAP', 'P0040005': 'WVAP', 'P0040006': 'BVAP', 'P0040007': 'AMINVAP',
               'P0040008': 'ASIANVAP', 'P0040009': 'NHPIVAP', 'P0040010': 'OTHERVAP', 'P0040011': '2MOREVAP'}

In [15]:
population_df.rename(columns=rename_dict, inplace = True)

In [16]:
population_df['H_WHITE'] = population_df.apply(lambda t: t['P0010003'] - t['NH_WHITE'], 1)
population_df['H_BLACK'] = population_df.apply(lambda t: t['P0010004'] - t['NH_BLACK'], 1)
population_df['H_AMIN'] = population_df.apply(lambda t: t['P0010005'] - t['NH_AMIN'], 1)
population_df['H_ASIAN'] = population_df.apply(lambda t: t['P0010006'] - t['NH_ASIAN'], 1)
population_df['H_NHPI'] = population_df.apply(lambda t: t['P0010007'] - t['NH_NHPI'], 1)
population_df['H_OTHER'] = population_df.apply(lambda t: t['P0010008'] - t['NH_OTHER'], 1)
population_df['H_2MORE'] = population_df.apply(lambda t: t['P0010009'] - t['NH_2MORE'], 1)

# Read the base vest data
Now using it as a "base precinct", but it could be vest 18 or vest 16 if vest 20 is not working

In [17]:
def add_vest_base(vest, start_col, year):
    original_col = vest.columns[start_col:-1]
    new_col = [rename(i, year) for i in original_col]
    rename_dict = dict(zip(original_col, new_col))
    vest = vest.rename(columns=rename_dict)
    vest = vest.groupby(level=0, axis=1).sum()
    vest = gpd.GeoDataFrame(vest, crs="EPSG:4269")
    
    return vest

### Check if vest 20 can be used as base

In [30]:
vest20 = gpd.read_file(vest20_data)

In [31]:
if maup.doctor(vest20) != True:
    vest20 = do_smart_repair(vest20)

100%|████████████████████████████████████████| 1764/1764 [00:08<00:00, 220.44it/s]


There are 29 overlaps.
Snapping all geometries to a grid with precision 10^( -3 ) to avoid GEOS errors.
Identifying overlaps...


100%|███████████████████████████████████████| 1810/1810 [00:01<00:00, 1652.64it/s]


Resolving overlaps...
Assigning order 2 pieces...
Filling gaps...


100%|████████████████████████████████████████| 1764/1764 [00:07<00:00, 243.03it/s]


### If it is true for maup doctor, we will use it as the base vest data.
Check where the election column starts, this should be the same for all vest data in that state

In [32]:
vest20.columns

Index(['STATEFP20', 'COUNTYFP20', 'VTDST20', 'GEOID20', 'NAME20', 'G20PRERTRU',
       'G20PREDBID', 'G20PRELJOR', 'G20PREGHAW', 'G20PREABLA', 'G20PREOCAR',
       'G20PREIWES', 'G20PREICOL', 'G20PREIPIE', 'G20USSRHYD', 'G20USSDESP',
       'G20USSLEDW', 'geometry'],
      dtype='object')

In [28]:
# maup.assign failed

## Parameters that need to be checked

In [33]:
start_col = 5
vest_base_data = vest20
year = '20'

In [34]:
vest_base = add_vest_base(vest_base_data, start_col, year)

In [35]:
# vap and population have the same GEOID20
blocks_to_precincts_assignment = maup.assign(population_df.geometry, vest_base.geometry)

100%|█████████████████████████████████████████| 1764/1764 [01:09<00:00, 25.55it/s]


In [36]:
pop_col = ['TOTPOP', 'HISP', 'NH_WHITE', 'NH_BLACK', 'NH_AMIN', 'NH_ASIAN', 'NH_NHPI', 'NH_OTHER', 'NH_2MORE', 'H_WHITE', 'H_BLACK', 'H_AMIN', 'H_ASIAN', 'H_NHPI', 'H_OTHER', 'H_2MORE', 'VAP', 'HVAP', 'WVAP', 'BVAP', 'AMINVAP', 'ASIANVAP', 'NHPIVAP', 'OTHERVAP', '2MOREVAP']

In [37]:
vest_base[pop_col] = population_df[pop_col].groupby(blocks_to_precincts_assignment).sum()

In [38]:
election_df = gpd.GeoDataFrame(vest_base, crs="EPSG:4269")

### Check if the population agrees

In [39]:
check_population(population_df, vest_base)

population agrees


# Add more vest data
### vest 19 (seems to be corrupted)

In [55]:
vest19 = gpd.read_file(vest19_data)
vest19.columns

DriverError: './ms_data/ms_vest_19/ms_vest_19.shp' not recognized as a supported file format.

### vest 18

In [40]:
vest18 = gpd.read_file(vest18_data)

In [41]:
vest18.columns

Index(['STATEFP18', 'COUNTYFP18', 'VTDST18', 'GEOID18', 'NAME18', 'G18USSRWIC',
       'G18USSDBAR', 'G18USSLBED', 'G18USSOOHA', 'S18USSRHYD', 'S18USSRMCD',
       'S18USSDESP', 'S18USSDBAR', 'R18USSRHYD', 'R18USSDESP', 'geometry'],
      dtype='object')

In [43]:
# check the result here
start_col = 5
election_df = add_vest(vest18, election_df, '18', population_df, start_col)

100%|████████████████████████████████████████| 1770/1770 [00:07<00:00, 224.13it/s]


There are 32 overlaps.
Snapping all geometries to a grid with precision 10^( -3 ) to avoid GEOS errors.
Identifying overlaps...


100%|███████████████████████████████████████| 1824/1824 [00:01<00:00, 1635.23it/s]


Resolving overlaps...
Assigning order 2 pieces...
Filling gaps...


100%|█████████████████████████████████████████| 1764/1764 [01:08<00:00, 25.94it/s]


population agrees


### vest 16

In [46]:
vest16 = gpd.read_file(vest16_data)
vest16.columns

Index(['STATEFP16', 'COUNTYFP16', 'VTDST16', 'GEOID16', 'NAME16', 'G16PRERTRU',
       'G16PREDCLI', 'G16PRELJOH', 'G16PREGSTE', 'G16PRECCAS', 'G16PREOHED',
       'G16PREOFUE', 'geometry'],
      dtype='object')

In [47]:
start_col = 5
election_df = add_vest(vest16, election_df, '16', population_df, start_col)

100%|████████████████████████████████████████| 1797/1797 [00:08<00:00, 212.16it/s]


There are 32 overlaps.
Snapping all geometries to a grid with precision 10^( -3 ) to avoid GEOS errors.
Identifying overlaps...


100%|███████████████████████████████████████| 1851/1851 [00:01<00:00, 1618.12it/s]


Resolving overlaps...
Assigning order 2 pieces...
Filling gaps...


100%|█████████████████████████████████████████| 1764/1764 [01:08<00:00, 25.85it/s]


population agrees


## Add the district data

In [48]:
cong_df = gpd.read_file(cd_data).to_crs('EPSG:4269')
send = gpd.read_file(send_data).to_crs('EPSG:4269')
hdist = gpd.read_file(hdist_data).to_crs('EPSG:4269')

In [49]:
cong_df.head()

,ID,AREA,DISTRICT,MEMBERS,LOCKED,NAME,POPULATION,WHITE,BLACK,F18_POP,...,DEVIATION,F_DEVIATIO,F_WHITE,F_BLACK,F_18_POP,F_18_WHT,F_18_BLK,MULTIPLE_F,DISTRICT_L,geometry
0,1,10096.88,2801,1.0,None,Trent Kelly,740319,483849,202327,570086,...,-1.0,-0.0,0.6536,0.2733,0.7701,0.6751,0.2632,2801|740319|26.32%,2801|-0%,"POLYGON ((-88.30443 33.28832, -88.30463 33.288..."
1,2,18413.48,2802,1.0,None,Bennie Thompson,740319,238483,468074,571715,...,-1.0,-0.0,0.3221,0.6323,0.7723,0.3482,0.6105,2802|740319|61.05%,2802|-0%,"POLYGON ((-90.04584 32.56758, -90.04610 32.568..."
2,3,11830.83,2803,1.0,None,Michael Guest,740320,442249,244759,568068,...,0.0,0.0,0.5974,0.3306,0.7673,0.6202,0.3179,2803|740320|31.79%,2803|0%,"POLYGON ((-90.45004 32.57378, -90.45004 32.573..."
3,4,8121.62,2804,1.0,None,Steven Palazzo,740321,494312,169321,567730,...,1.0,0.0,0.6677,0.2287,0.7669,0.6914,0.2171,2804|740321|21.71%,2804|0%,"POLYGON ((-89.72812 31.00230, -89.72707 31.002..."


In [50]:
send.head()

,ID,AREA,DISTRICT,MEMBERS,LOCKED,NAME,POPULATION,WHITE,BLACK,F18_POP,...,DISTRICT_L,MULTIPLE_F,MULTIPLE_1,OTH,LABEL,Distnum,Sen_Shade,Shade_Num,Dist20,geometry
0,1,190.37,28001,1.0,None,None,56991,36210,14991,42710,...,28001|0.08%,01|56991|43|24.72%,28001|56991|0.08%|24.72%,2399,1.0,1,1,12,28001,"POLYGON ((-90.20020 34.72442, -90.20030 34.724..."
1,2,52.41,28002,1.0,None,None,57640,31696,19683,43422,...,28002|1.22%,02|57640|692|32.02%,28002|57640|1.22%|32.02%,4006,2.0,2,2,2,28002,"POLYGON ((-90.03333 34.97668, -90.03335 34.976..."
2,3,1078.91,28003,1.0,None,None,59005,40050,14414,45381,...,28003|3.61%,"03|59005|2,057|24.8%",28003|59005|3.61%|24.8%,2670,3.0,3,3,9,28003,"POLYGON ((-89.35286 34.91725, -89.35285 34.921..."
3,4,861.39,28004,1.0,None,None,56555,44798,7801,43590,...,28004|-0.69%,04|56555|-393|13.48%,28004|56555|-0.69%|13.48%,1729,4.0,4,2,15,28004,"POLYGON ((-89.08849 34.59825, -89.08849 34.598..."
4,5,1131.18,28005,1.0,None,None,58670,51866,3930,45880,...,28005|3.02%,"05|58670|1,722|6.72%",28005|58670|3.02%|6.72%,1052,5.0,5,13,3,28005,"POLYGON ((-88.36547 34.75560, -88.36569 34.755..."


In [51]:
hdist.head()

,ID,AREA,DISTRICT,MEMBERS,LOCKED,NAME,POPULATION,WHITE,BLACK,F18_POP,...,F_BLACK,F_18_POP,F_18_WHT,F_18_BLK,MULTIPLE_F,F_18_AP_BL,DISTRICT_L,Distnum,Shade_Num,geometry
0,1,464.05,28001,1.0,None,None,23965,21988,670,18942,...,0.0280,0.7904,0.9232,0.0289,28001|23965|-308|-1.27%,0.0334,28001|-1.27%,1,1,"POLYGON ((-88.48808 34.93068, -88.48808 34.930..."
1,2,216.59,28002,1.0,None,None,23089,17424,3898,17753,...,0.1688,0.7689,0.7811,0.1622,"28002|23089|-1,184|-4.88%",0.1678,28002|-4.88%,2,2,"POLYGON ((-88.52850 34.99583, -88.52841 34.995..."
2,3,437.35,28003,1.0,None,None,25373,21604,2466,19694,...,0.0972,0.7762,0.8596,0.0980,"28003|25373|1,100|4.53%",0.1039,28003|4.53%,3,3,"POLYGON ((-88.33875 34.46377, -88.33891 34.463..."
3,4,524.45,28004,1.0,None,None,23122,18043,3488,17768,...,0.1509,0.7684,0.7978,0.1498,"28004|23122|-1,151|-4.74%",0.1555,28004|-4.74%,4,14,"POLYGON ((-89.01738 34.91016, -89.01737 34.911..."
4,5,576.05,28005,1.0,None,None,23345,8234,13635,18730,...,0.5841,0.8023,0.3614,0.5861,28005|23345|-928|-3.82%,0.5971,28005|-3.82%,5,11,"POLYGON ((-89.63581 34.87547, -89.63580 34.876..."


In [52]:
election_df = add_district(cong_df, "CD", election_df, "DISTRICT")

100%|███████████████████████████████████████████████| 4/4 [00:04<00:00,  1.15s/it]


In [53]:
election_df = add_district(send, "SEND", election_df, "DISTRICT")

100%|█████████████████████████████████████████████| 52/52 [00:11<00:00,  4.49it/s]


In [54]:
election_df = add_district(hdist, "HDIST", election_df, "DISTRICT")

100%|███████████████████████████████████████████| 122/122 [00:12<00:00, 10.07it/s]


In [60]:
election_df.columns

Index(['2MOREVAP', 'AMINVAP', 'ASIANVAP', 'BVAP', 'COUNTYFP20', 'GEOID20',
       'HISP', 'HVAP', 'H_2MORE', 'H_AMIN', 'H_ASIAN', 'H_BLACK', 'H_NHPI',
       'H_OTHER', 'H_WHITE', 'NAME20', 'NHPIVAP', 'NH_2MORE', 'NH_AMIN',
       'NH_ASIAN', 'NH_BLACK', 'NH_NHPI', 'NH_OTHER', 'NH_WHITE', 'OTHERVAP',
       'PRE16D', 'PRE16O', 'PRE16R', 'PRE20D', 'PRE20O', 'PRE20R', 'STATEFP20',
       'TOTPOP', 'USS18D', 'USS18O', 'USS18R', 'USS20D', 'USS20O', 'USS20R',
       'VAP', 'VTDST20', 'WVAP', 'geometry', 'CD', 'SEND', 'HDIST'],
      dtype='object')

### Put the base precinct year after the precinct information column

In [61]:
base_columns = {}
if 'COUNTYFP' + year not in election_df.columns:
    base_columns = {
        'STATEFP':'STATEFP'+year,
        'COUNTYFP':'COUNTYFP'+year,
        'PRECINCT':'PRECINCT'+year,
        'GEOID':'GEOID'+year,
        'NAME':'NAME'+year,
        'VTDST':'VTDST'+year
    }
election_df.rename(columns=base_columns, inplace = True)

In [63]:
# reorder the columns
fixed_columns = [
    'STATEFP'+year,
    'COUNTYFP'+year,
    # 'PRECINCT'+year,
    'GEOID'+year,
    'NAME'+year,
    'VTDST'+year,
    'CD',
    'SEND',
    'HDIST',
    'TOTPOP',
    'NH_2MORE',
    'NH_AMIN',
    'NH_ASIAN',
    'NH_BLACK',
    'NH_NHPI',
    'NH_OTHER',
    'NH_WHITE',
    'HISP',
    'H_AMIN',
    'H_ASIAN',
    'H_BLACK',
    'H_NHPI',
    'H_OTHER',
    'H_WHITE',
    'H_2MORE',
    'VAP',
    'HVAP',
    'WVAP',
    'BVAP',
    'AMINVAP',
    'ASIANVAP',
    'NHPIVAP',
    'OTHERVAP',
    '2MOREVAP']

election_columns = [col for col in election_df.columns if col not in fixed_columns]
final_col = fixed_columns + election_columns
election_df = election_df[final_col]

In [64]:
list(election_df.columns)

['STATEFP20',
 'COUNTYFP20',
 'GEOID20',
 'NAME20',
 'VTDST20',
 'CD',
 'SEND',
 'HDIST',
 'TOTPOP',
 'NH_2MORE',
 'NH_AMIN',
 'NH_ASIAN',
 'NH_BLACK',
 'NH_NHPI',
 'NH_OTHER',
 'NH_WHITE',
 'HISP',
 'H_AMIN',
 'H_ASIAN',
 'H_BLACK',
 'H_NHPI',
 'H_OTHER',
 'H_WHITE',
 'H_2MORE',
 'VAP',
 'HVAP',
 'WVAP',
 'BVAP',
 'AMINVAP',
 'ASIANVAP',
 'NHPIVAP',
 'OTHERVAP',
 '2MOREVAP',
 'PRE16D',
 'PRE16O',
 'PRE16R',
 'PRE20D',
 'PRE20O',
 'PRE20R',
 'USS18D',
 'USS18O',
 'USS18R',
 'USS20D',
 'USS20O',
 'USS20R',
 'geometry']

In [65]:
# store the result in directory "il"
os.makedirs("./{}".format(state_ab))
election_df.to_file("./{}/{}.shp".format(state_ab, state_ab))
election_df.to_file('./{}/{}.geojson'.format(state_ab, state_ab), driver='GeoJSON')

# Only do once to build json and read from file when generating ensembles
graph = Graph.from_file("./{}/{}.shp".format(state_ab, state_ab), ignore_errors=True)
graph.to_json("./{}/{}.json".format(state_ab, state_ab))